In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
raw_data_dir = "/content/drive/MyDrive/CS 224N Project/Evaluation/Data/Processed_data/Train/"
raw_testdata_dir = "/content/drive/MyDrive/CS 224N Project/Evaluation/Data/Processed_data/Test/"
file_names = ['abortion.csv', 'before_iraq.csv', 'firearms.csv', 'main_iraq.csv']

In [ ]:
def get_data(file_name, data_dir=raw_data_dir):
  data = pd.read_csv(data_dir + file_name, index_col=None)
  np.shape(data)
  return data


def get_test_data(file_name, data_dir=raw_testdata_dir):
  data = pd.read_csv(data_dir + file_name, index_col=None)
  np.shape(data)
  return data

In [ ]:
train_abortion_df = get_data('abortion_data.csv')
train_firearms_df = get_data('firearms_data.csv')
train_main_iraq_df = get_data('main_iraq_data.csv')
test_abortion_df = get_test_data('abortion_data.csv')
test_firearms_df = get_test_data('firearms_data.csv')
test_main_iraq_df = get_test_data('main_iraq_data.csv')

In [ ]:
train_abortion_label_df = get_data('abortion_labels.csv')
train_firearms_label_df = get_data('firearms_labels.csv')
train_main_iraq_label_df = get_data('main_iraq_labels.csv')
test_abortion_label_df = get_test_data('abortion_labels.csv')
test_firearms_label_df = get_test_data('firearms_labels.csv')
test_main_iraq_label_df = get_test_data('main_iraq_labels.csv')

In [ ]:
from sklearn.model_selection import train_test_split

train_main_iraq, val_main_iraq, train_main_iraq_label, val_main_iraq_label = train_test_split(
  train_main_iraq_df, train_main_iraq_label_df, test_size=0.125, random_state=42
)

train_main_iraq = train_main_iraq.reset_index(drop=True)
val_main_iraq = val_main_iraq.reset_index(drop=True)
train_main_iraq_label = train_main_iraq_label.reset_index(drop=True)
val_main_iraq_label = val_main_iraq_label.reset_index(drop=True)

In [ ]:
!pip install lightning wandb peft

In [ ]:
import torch
from transformers import BertTokenizer
from lightning.pytorch import LightningDataModule

class my_dataset(torch.utils.data.Dataset):
  def __init__(self, tokenizer, speech_data, label_data):
    self.tokenizer = tokenizer
    tokenizer.bos_token = tokenizer.cls_token
    tokenizer.eos_token = tokenizer.sep_token
    self.tokenizer.pad_token = tokenizer.eos_token
    self.speech = speech_data
    self.label = label_data
    self.label_to_id = {'R': [0], 'D': [1]}

  def __len__(self):
    return len(self.speech)

  def __getitem__(self, index):
    text = self.tokenizer.bos_token + self.speech['speech'][index] + self.tokenizer.eos_token
    input_ids = self.tokenizer.encode(text, return_tensors='pt', padding='max_length', truncation=True, max_length=512)[0]
    pad_id = 2
    attention_mask = torch.Tensor([1 if el != pad_id else 0 for el in input_ids]).to(dtype=torch.long)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'label': self.label_to_id [self.label['party'][index]]
    }


class DataModule(LightningDataModule):
    def __init__(
            self,
            model_type,
            batch_size,
            train_speech_df,
            train_label_df,
            valid_speech_df,
            valid_label_df,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.train_speech_df = train_speech_df
        self.train_label_df = train_label_df
        self.valid_speech_df = valid_speech_df
        self.valid_label_df = valid_label_df
        self.tokenizer = AutoTokenizer.from_pretrained(model_type)

    def prepare_data(self):
      pass

    def setup(self, stage: str):

        self.train_ds = my_dataset(self.tokenizer, self.train_speech_df, self.train_label_df)
        self.valid_ds = my_dataset(self.tokenizer, self.valid_speech_df, self.valid_label_df)


    def train_dataloader(self):
        loader = torch.utils.data.DataLoader(self.train_ds, batch_size=self.batch_size, drop_last=True, pin_memory=True)
        return loader

    def val_dataloader(self):
        loader = torch.utils.data.DataLoader(self.valid_ds, batch_size=self.batch_size, drop_last=True, pin_memory=True)
        return loader


In [ ]:
import os
from torch import optim, nn, utils, Tensor
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import lightning as L

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from peft import get_peft_config, get_peft_model, PeftModel, PeftConfig, LoraConfig, TaskType
import torch.nn.functional as F

from torch.optim.lr_scheduler import CosineAnnealingLR

import wandb
from lightning.pytorch.loggers import WandbLogger

class BertModule(L.LightningModule):
    def __init__(self, use_lora=False):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModel.from_pretrained('google-bert/bert-base-uncased')
        self.proj = nn.Linear(768, 1)
        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad) + sum(p.numel() for p in self.proj.parameters() if p.requires_grad)
        for name, param in self.model.named_parameters():
          param.requires_grad_(False)

        bert_dim = 768

        self.loss_func = F.binary_cross_entropy_with_logits
        self.valid_preds = []
        self.valid_labels = []


    def training_step(self, batch, batch_idx):
        #labels = torch.stack(batch['label']).T.to(dtype=torch.float16)
        labels = torch.stack(batch['label']).T.to(dtype=torch.float16).view(-1, 1)
        del batch['label']
        bert_logits = self.model(**batch).last_hidden_state[:, 0]
        logits = self.proj(bert_logits)
        loss = self.loss_func(logits, labels)
        self.log_dict({'loss': loss}, prog_bar=True)

        lr = self.trainer.optimizers[0].param_groups[0]['lr']
        self.log('lr', lr, prog_bar=True)

        return {'loss': loss}

    def validation_step(self, batch, batch_idx):
        #labels = torch.stack(batch['label'])
        labels = torch.stack(batch['label']).T.to(dtype=torch.float16).view(-1, 1)
        del batch['label']
        bert_logits = self.model(**batch).last_hidden_state[:, 0]
        logits = self.proj(bert_logits)
        val_loss = self.loss_func(logits, labels.to(dtype=torch.float16))
        preds = torch.round(F.sigmoid(logits))
        self.valid_preds.append(preds)
        self.valid_labels.append(labels)
        self.log('val_loss', val_loss, prog_bar=True)
        return {'val_loss': val_loss}

    def on_validation_epoch_end(self):
        TP, TN, FP, FN = 0, 0, 0, 0
        valid_preds = torch.stack(self.valid_preds).view(-1)
        valid_labels = torch.stack(self.valid_labels).view(-1)
        for pred, gold in zip(valid_preds, valid_labels):
          if pred == 1:
            if gold == 1:
              TN += 1
            else:
              FP += 1
          else:
            if gold == 1:
              FN += 1
            else:
              TN += 1

        recall = TN / (TN + FP)
        precision = TN / (TN + FN)
        acc = (TN + TP) / (TN + FP + FN + TP)
        f1 = 2 * recall * precision / (recall + precision)
        print('Recall', recall)
        print('Precision', precision)
        print('F1', f1)
        print('Accuracy', acc)
        self.log_dict({'Recall': recall,
                       'Accuracy': acc,
                       'Precision': precision,
                       'F1': f1,
                       }, prog_bar=True)
        self.valid_preds.clear()
        self.valid_labels.clear()


    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=3e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=10, eta_min=1e-6)
        sch_config = {
            "scheduler": scheduler,
            "interval": "step"
        }
        return {"optimizer": optimizer, "lr_scheduler": sch_config}

    def get_progress_bar_dict(self):
        items = super().get_progress_bar_dict()
        items.pop("v_num", None)
        return items

In [ ]:
import os
# this likely needs to be switched too depending on your set up
# Define the path to save model checkpoints in Google Drive
checkpoint_dir = '/content/drive/MyDrive/ModelCheckpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

### Training/Testing

In [ ]:
import time
from lightning.pytorch.callbacks import ModelCheckpoint

test_data = test_main_iraq_df
test_labels = test_main_iraq_label_df
model_name="google-bert/bert-base-uncased"
batch_size=16

#Abortion training (without finetuning on Iraq)
bert_module_abortion_initial = BertModule(use_lora=True)
timestamp = time.strftime("%Y%m%d-%H%M%S")

checkpoint_callback_abortion_initial = ModelCheckpoint(
    monitor='val_loss',
    dirpath=checkpoint_dir,
    filename=f'bert-initial-{timestamp}-{{epoch:02d}}-{{val_loss:.2f}}',
    save_top_k=1,
    mode='min'
)

trainer_abortion_initial = L.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=10,
    num_sanity_val_steps=0,
    callbacks=[checkpoint_callback_abortion_initial]
)

trainer_abortion_initial.fit(model=bert_module_abortion_initial, datamodule=DataModule(model_name, batch_size, train_abortion_df, train_abortion_label_df, test_abortion_df, test_abortion_label_df))

# write name of the path here for saved model
initial_best_model_path = checkpoint_callback_abortion_initial.best_model_path

bert_module_eval = BertModule.load_from_checkpoint(initial_best_model_path, use_lora=use_lora)
initial_results = trainer_abortion_initial.validate(model=bert_module_eval, datamodule=DataModule(model_name, batch_size, None, None, test_data, test_labels))
print(initial_results)





In [ ]:
#Abortion training (finetuning on Iraq)
test_data = test_main_iraq_df
test_labels = test_main_iraq_label_df

use_lora=True
model_name="google-bert/bert-base-uncased"
batch_size=16

timestamp = time.strftime("%Y%m%d-%H%M%S")

# this likely needs to be switched too depending on your set up
checkpoint_dir = '/content/drive/MyDrive/ModelCheckpoints'

# type in the initial best model path from before
initial_best_model_path = ''

bert_module_finetune = BertModule.load_from_checkpoint(initial_best_model_path, use_lora=use_lora)
checkpoint_callback_finetune = ModelCheckpoint(
          monitor='val_loss',
          dirpath=checkpoint_dir,
          filename=f'bert-finetune-{timestamp}-{{epoch:02d}}-{{val_loss:.2f}}',
          save_top_k=1,
          mode='min'
      )

trainer_finetune = L.Trainer(
          devices=1,
          accelerator="gpu",
          max_epochs=10,
          num_sanity_val_steps=0,
          callbacks=[checkpoint_callback_finetune]
)

trainer_finetune.fit(model=bert_module_finetune, datamodule=DataModule(model_name, batch_size, train_main_iraq, train_main_iraq_label, val_main_iraq, val_main_iraq_label))
finetune_results = trainer_finetune.validate(model=deberta_module_finetune, datamodule=DataModule(model_name, batch_size, None, None, test_data, test_labels))
print(finetune_results)